In [1]:
# Download dependencies
import csv
import os
import re
import time
from collections import deque
from pathlib import Path
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup

In [2]:
BASE = "https://www.wikiart.org"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
}


def slugify(value: str) -> str:
    value = re.sub(r"[^a-zA-Z0-9]+", "-", value.strip().lower())
    return value.strip("-") or "artwork"


def normalize_url(url: str) -> str:
    parsed = urlparse(url)
    return f"{parsed.scheme}://{parsed.netloc}{parsed.path}".rstrip("/")


def looks_like_artwork_url(url: str) -> bool:
    path = urlparse(url).path.strip("/")
    parts = path.split("/")
    return len(parts) >= 3 and parts[0] == "en"


class WikiArtCrawler:
    def __init__(self, out_dir="wikiart_crawl", delay=1.0, timeout=30):
        self.out_dir = Path(out_dir)
        self.images_dir = self.out_dir / "images"
        self.out_dir.mkdir(parents=True, exist_ok=True)
        self.images_dir.mkdir(parents=True, exist_ok=True)
        self.delay = delay
        self.timeout = timeout
        self.session = requests.Session()
        self.session.headers.update(HEADERS)
        self.visited_pages = set()
        self.seen_artworks = set()

    def get_soup(self, url: str) -> BeautifulSoup:
        response = self.session.get(url, timeout=self.timeout)
        response.raise_for_status()
        return BeautifulSoup(response.text, "html.parser")

    def extract_meta_table(self, soup: BeautifulSoup) -> dict:
        data = {}
        for row in soup.select('.additional-information-item'):
            label = row.select_one('.additional-information-item-label')
            value = row.select_one('.additional-information-item-value')
            if label and value:
                key = re.sub(r'\s+', ' ', label.get_text(' ', strip=True)).rstrip(':').lower()
                val = re.sub(r'\s+', ' ', value.get_text(' ', strip=True))
                data[key] = val
        return data

    def scrape_artwork(self, url: str, download_image: bool = True) -> dict:
        soup = self.get_soup(url)
        meta = self.extract_meta_table(soup)

        title_el = soup.select_one('h3') or soup.select_one('h1')
        title = title_el.get_text(' ', strip=True) if title_el else ''

        artist = ''
        artist_url = ''
        author_meta = soup.select_one('meta[property="og:title"]')
        if author_meta and ' by ' in author_meta.get('content', ''):
            maybe = author_meta['content'].split(' by ')[-1]
            artist = maybe.split(' - ')[0].strip()

        canonical = soup.select_one('link[rel="canonical"]')
        source_url = canonical['href'] if canonical and canonical.has_attr('href') else url

        img_url = ''
        img = soup.select_one('meta[property="og:image"]')
        if img and img.has_attr('content'):
            img_url = img['content']

        description = ''
        desc = soup.select_one('meta[property="og:description"]')
        if desc and desc.has_attr('content'):
            description = desc['content']

        tags = []
        for a in soup.select('a[href*="/en/paintings-by-tag/"]'):
            txt = a.get_text(' ', strip=True)
            if txt and txt not in tags:
                tags.append(txt)

        artist_link = soup.select_one('a[href^="/en/"]')
        if artist_link and artist_link.has_attr('href') and len(urlparse(artist_link['href']).path.strip('/').split('/')) == 2:
            artist_url = urljoin(BASE, artist_link['href'])
            if not artist:
                artist = artist_link.get_text(' ', strip=True)

        record = {
            'source_url': source_url,
            'title': title,
            'artist': artist,
            'artist_url': artist_url,
            'year': meta.get('date', ''),
            'style': meta.get('style', '') or meta.get('styles', ''),
            'genre': meta.get('genre', ''),
            'media': meta.get('media', ''),
            'location': meta.get('location', ''),
            'dimensions': meta.get('dimensions', ''),
            'description': description,
            'tags': '|'.join(tags),
            'image_url': img_url,
            'local_image_path': '',
        }

        if download_image and img_url:
            ext = os.path.splitext(img_url.split('?')[0])[1] or '.jpg'
            filename = f"{slugify(artist)}-{slugify(title)}{ext}"
            filepath = self.images_dir / filename
            if not filepath.exists():
                r = self.session.get(img_url, timeout=self.timeout)
                r.raise_for_status()
                filepath.write_bytes(r.content)
                time.sleep(self.delay)
            record['local_image_path'] = str(filepath)

        return record

    def discover_links(self, soup: BeautifulSoup) -> list[str]:
        links = []
        for a in soup.select('a[href]'):
            href = a['href']
            full = urljoin(BASE, href)
            normalized = normalize_url(full)
            if normalized.startswith(BASE + '/en/'):
                links.append(normalized)
        return list(dict.fromkeys(links))

    def crawl(self, seed_urls, max_artworks=100, max_pages=300, download_images=True, csv_name='artworks.csv'):
        queue = deque(normalize_url(u) for u in seed_urls)
        csv_path = self.out_dir / csv_name
        fieldnames = [
            'source_url', 'title', 'artist', 'artist_url', 'year', 'style', 'genre',
            'media', 'location', 'dimensions', 'description', 'tags', 'image_url', 'local_image_path'
        ]

        with csv_path.open('w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()

            while queue and len(self.visited_pages) < max_pages and len(self.seen_artworks) < max_artworks:
                current = queue.popleft()
                if current in self.visited_pages:
                    continue
                self.visited_pages.add(current)

                try:
                    soup = self.get_soup(current)
                except Exception as exc:
                    print(f"Failed page: {current} -> {exc}")
                    continue

                if looks_like_artwork_url(current):
                    try:
                        record = self.scrape_artwork(current, download_image=download_images)
                        if record['source_url'] not in self.seen_artworks and record['title']:
                            writer.writerow(record)
                            self.seen_artworks.add(record['source_url'])
                            print(f"Saved artwork {len(self.seen_artworks)}: {record['title']}")
                    except Exception as exc:
                        print(f"Failed artwork: {current} -> {exc}")

                for link in self.discover_links(soup):
                    if link not in self.visited_pages and link not in queue:
                        queue.append(link)

                time.sleep(self.delay)

        return csv_path

In [3]:
seed_urls = [
    'https://www.wikiart.org/en/paintings-by-style/impressionism',
    'https://www.wikiart.org/en/paintings-by-genre/landscape',
    'https://www.wikiart.org/en/vincent-van-gogh',
]

crawler = WikiArtCrawler(out_dir='wikiart_crawl', delay=1.5)
csv_path = crawler.crawl(
    seed_urls=seed_urls,
    max_artworks=50,
    max_pages=200,
    download_images=True,
    csv_name='wikiart_artworks.csv',
)
print(f'Dataset saved to: {csv_path}')

Failed artwork: https://www.wikiart.org/en/paintings-by-style/impressionism -> 522 Server Error: <none> for url: https://uploads.wikiart.org/Content/wiki/img/favicon-256x256.png
Failed artwork: https://www.wikiart.org/en/paintings-by-genre/landscape -> 522 Server Error: <none> for url: https://uploads.wikiart.org/Content/wiki/img/favicon-256x256.png
Saved artwork 1: Houses at Argenteuil
Failed artwork: https://www.wikiart.org/en/actionhistory/viewbydocumentid/57726b51edc2cb3880ad7600 -> 522 Server Error: <none> for url: https://uploads.wikiart.org/Content/wiki/img/favicon-256x256.png
Saved artwork 2: Invincible [2024]
Failed artwork: https://www.wikiart.org/en/Alphabet/a -> 522 Server Error: <none> for url: https://uploads.wikiart.org/Content/wiki/img/favicon-256x256.png
Failed artwork: https://www.wikiart.org/en/popular-paintings/alltime -> 523 Server Error: <none> for url: https://uploads.wikiart.org/Content/wiki/img/favicon-256x256.png
Saved artwork 3: Byzantine Faces
Saved artwork 

KeyboardInterrupt: 